# Quanto custa morar em Lucas do Rio Verde (MT): o mercado imobiliário por dentro

**Autora:** Thamires Martins da Silva  
**Fonte dos dados:** Anúncios de venda de 13 imobiliárias e portais locais + Índice FipeZap (contexto regional)  
**Ferramentas:** Python (pandas, matplotlib)

---

## Contexto

Este é o **terceiro projeto de uma série de quatro análises conectadas** sobre habitação em Lucas do Rio Verde:

- **Camada 1** — estima cenários de demanda habitacional a partir do crescimento populacional.
- **Camada 2** — mede a oferta registrada de habitação de interesse social.
- **Camada 3 (este notebook)** — descreve os preços anunciados de imóveis, por m², bairro e tipo.
- **Camada 4** — compara a amostra de preços com faixas de renda de referência em cenários de acessibilidade.

A sequência ajuda a contextualizar a questão habitacional local. Ela organiza perguntas e hipóteses, mas não prova causalidade entre crescimento populacional, oferta, preços e acessibilidade.

## Pergunta de negócio

**Quanto custam os imóveis anunciados em Lucas do Rio Verde, como o preço por m² varia entre bairros e tipos de imóvel, e como essa amostra se compara a Cuiabá e ao índice FipeZap?**

## Sobre os dados

Não existe índice público de preço imobiliário específico para Lucas do Rio Verde — o FipeZap cobre um conjunto limitado de cidades (em Mato Grosso, Cuiabá). Por isso, o dataset foi construído a partir de anúncios de venda coletados em sites de imobiliárias locais, com preço, área construída e bairro.

A base publicada tem **233 registros de 25 bairros**, de 13 fontes. Três linhas integralmente duplicadas foram removidas. Combinações iguais de bairro, tipo, área e preço em fontes diferentes permanecem como candidatas para revisão manual; elas não são removidas automaticamente sem identificação suficiente do imóvel.

**Critério de qualidade na coleta:** o preço por m² só faz sentido sobre a **área construída**. Fontes que informavam apenas área de terreno foram descartadas; da fonte cujos cards misturavam área construída e terreno nas casas, entraram apenas os apartamentos, nos quais a área privativa representa a área construída do imóvel.

## Limitações assumidas

1. **Retrato do momento (outubro de 2026), não série histórica:** anúncios antigos não ficam necessariamente arquivados.
2. **Preço pedido, não preço fechado:** o anúncio não comprova o valor final de transação.
3. **Possível sobreposição entre fontes:** imobiliárias podem anunciar o mesmo imóvel. A base elimina duplicatas integrais e sinaliza combinações similares para revisão, mas a identificação de imóveis espelhados exige mais evidência.
4. **Área declarada na fonte:** não é uma medição independente.
5. **Comparação regional de fontes distintas:** Lucas usa anúncios locais; Cuiabá e Brasil usam FipeZap. A comparação indica ordem de grandeza, não uma medição perfeitamente equivalente.

## 1. Configuração e carga

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

AZUL_ESC='#1a3a5c'; AZUL='#2c7fb8'; TEAL='#7fcdbb'; LARANJA='#e8703a'; CINZA='#8a9299'

df = pd.read_csv(Path.cwd()/'precos_imoveis_lucas.csv', sep=';')

# Verifica duplicatas exatas e remove-as como proteção adicional.
duplicatas_exatas = int(df.duplicated().sum())
if duplicatas_exatas:
    df = df.drop_duplicates().copy()

# Combinações iguais podem corresponder ao mesmo imóvel em fontes diferentes.
# São sinalizadas para revisão manual, sem remoção automática.
chave_candidata = ['bairro', 'tipo', 'area_m2', 'preco_total', 'preco_m2']
candidatos_duplicacao = df[df.duplicated(subset=chave_candidata, keep=False)].sort_values(chave_candidata)

print(f'Duplicatas exatas removidas ao carregar: {duplicatas_exatas}')
print(f'Total de anúncios após verificação: {len(df)}')
print(f'Registros com possível sobreposição entre fontes: {len(candidatos_duplicacao)}')
if not candidatos_duplicacao.empty:
    print(candidatos_duplicacao[chave_candidata + ['fonte']].to_string(index=False))
print(f'Bairros nomeados: {df[df["bairro"]!="Não informado"]["bairro"].nunique()}')
print(f'Fontes: {df["fonte"].value_counts().to_dict()}')
print()
print(df[['bairro','tipo','area_m2','preco_total','preco_m2','fonte']].head(6).to_string(index=False))

## 2. Tratamento: separando o mercado típico

Imóveis **atípicos** — mansões de alto padrão e micro-apartamentos de investimento — distorcem a média e não representam o que uma família comum encontra. A análise principal usa só os típicos.

**Qualidade da base:** três linhas inteiras repetidas (mesmo bairro, tipo, área, preço, fonte e data) foram removidas antes desta versão pública. O arquivo publicado tem 233 registros. Combinações iguais de bairro/tipo/área/preço em fontes diferentes ficam como candidatas para inspeção manual; não são removidas automaticamente, pois podem ser anúncios espelhados ou imóveis distintos com características iguais.

In [ ]:
df['atip'] = df['atipico']=='sim'
tip = df[~df['atip']].copy()
print(f'Total:    {len(df)}')
print(f'  Tipicos:  {len(tip)}  (analise principal)')
print(f'  Atipicos: {df["atip"].sum()}  (luxo e investimento, separados)')
print()
print('--- Qualidade (tipicos) ---')
print(f'Nulos preco_m2: {tip["preco_m2"].isna().sum()}')
print(f'Area: {tip["area_m2"].min():.0f} a {tip["area_m2"].max():.0f} m2')
print(f'Casas: {(tip["tipo"]=="casa").sum()} | Apartamentos: {(tip["tipo"]=="apartamento").sum()}')

## 3. Quanto custa o metro quadrado em Lucas

In [ ]:
pm2 = tip['preco_m2']
print('=== PRECO POR M2 (tipicos) ===')
print(f'  Media:   R$ {pm2.mean():>6,.0f}/m2'.replace(',','.'))
print(f'  Mediana: R$ {pm2.median():>6,.0f}/m2'.replace(',','.'))
print(f'  Min-Max: R$ {pm2.min():,.0f} a R$ {pm2.max():,.0f}'.replace(',','.'))

In [ ]:
fig,ax=plt.subplots(figsize=(11,5.5))
ax.hist(pm2,bins=14,color=AZUL,edgecolor='white',zorder=3)
ax.axvline(pm2.median(),color=LARANJA,linestyle='--',linewidth=2.5,zorder=4,
           label=f'Mediana: R$ {pm2.median():,.0f}/m2'.replace(',','.'))
ax.set_title(f'Distribuicao do preco por m2 em Lucas do Rio Verde\nImoveis tipicos (n={len(pm2)})',
             fontsize=13,color=AZUL_ESC,pad=15)
ax.set_xlabel('Preco por m2 (R$)'); ax.set_ylabel('Numero de imoveis'); ax.legend(fontsize=11)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.grid(True,axis='y',alpha=.25,zorder=0)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,p:f'{int(x):,}'.replace(',','.')))
plt.tight_layout(); plt.show()

O metro quadrado em Lucas do Rio Verde gira em torno de **R\$ 6.212 (mediana)**, com a metade central dos anúncios típicos entre cerca de **R\$ 4.570 e R\$ 7.478/m²**. É um patamar alto para uma cidade de porte médio no interior — a seção 6 mostra a comparação com a capital.

## 4. O mapa de preços: quanto custa cada bairro

Com 25 bairros na amostra, dá para traçar a geografia de preços da cidade. Mostramos os bairros com pelo menos 5 anúncios, para que a média de cada um tenha robustez (anúncios sem bairro informado ficam de fora deste recorte).

In [ ]:
por_bairro = tip[tip['bairro']!='Não informado'].groupby('bairro')['preco_m2'].agg(['count','mean']).reset_index()
por_bairro = por_bairro[por_bairro['count']>=5].sort_values('mean',ascending=False)
print('=== PRECO/M2 POR BAIRRO (5+ anuncios) ===')
for _,r in por_bairro.iterrows():
    print(f'  {r["bairro"]:22} n={int(r["count"]):2}  R$ {r["mean"]:>6,.0f}/m2'.replace(',','.'))

In [ ]:
fig,ax=plt.subplots(figsize=(11,6.5))
pb = por_bairro.sort_values('mean')
cores=[LARANJA if v==pb['mean'].max() else (TEAL if v==pb['mean'].min() else AZUL) for v in pb['mean']]
ax.barh(pb['bairro'],pb['mean'],color=cores,zorder=3)
for i,(v,n) in enumerate(zip(pb['mean'],pb['count'])):
    ax.annotate(f'R$ {v:,.0f}'.replace(',','.'),(v,i),textcoords='offset points',
                xytext=(5,0),va='center',fontsize=10,fontweight='bold',color=AZUL_ESC)
ax.set_title('O mapa de precos de Lucas do Rio Verde\nPreco medio por m2, por bairro',
             fontsize=13,color=AZUL_ESC,pad=15)
ax.set_xlabel('Preco por m2 (R$)')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.grid(True,axis='x',alpha=.25,zorder=0)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,p:f'{int(x):,}'.replace(',','.')))
plt.tight_layout(); plt.show()

A geografia de preços é nítida. No topo, **Parque das Emas (~R\$ 8.000/m²), Bandeirantes (~R\$ 7.700/m²) e Parque dos Buritis (~R\$ 7.200/m²)**. Na base, **Jardim Imperial, Jardim Primavera, Cidade Nova e Rio Verde (~R\$ 3.600 a 4.250/m²)**.

A diferença entre as médias dos extremos é de aproximadamente **119% em relação ao bairro mais barato**. O recorte inclui apenas bairros com cinco ou mais anúncios e deve ser interpretado como indicativo, não como um índice oficial de preços.

## 5. Casa ou apartamento, e o que cabe no orçamento

In [ ]:
pt = tip.groupby('tipo')['preco_m2'].agg(['count','mean','median'])
print('=== PRECO/M2 POR TIPO ===')
for t,r in pt.iterrows():
    print(f'  {t.capitalize():12} n={int(r["count"]):2}  media R$ {r["mean"]:,.0f}  mediana R$ {r["median"]:,.0f}'.replace(',','.'))
print()
pr = tip['preco_total']
print('=== FAIXA DE PRECO TOTAL ===')
print(f'  Mais acessivel: R$ {pr.min():>10,.0f}'.replace(',','.'))
print(f'  Mediana:        R$ {pr.median():>10,.0f}'.replace(',','.'))
print(f'  Mais caro:      R$ {pr.max():>10,.0f}'.replace(',','.'))

In [ ]:
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(14,5.5))
# Esq: casa vs apto
tipos=['casa','apartamento']; medias=[pt.loc[t,'mean'] for t in tipos]
ax1.bar(['Casa','Apartamento'],medias,color=[TEAL,AZUL],width=.5,zorder=3)
for x,y in enumerate(medias):
    ax1.annotate(f'R$ {y:,.0f}'.replace(',','.'),(x,y),textcoords='offset points',
                 xytext=(0,8),ha='center',fontsize=12,fontweight='bold',color=AZUL_ESC)
ax1.set_title('Apartamento custa mais por m2',fontsize=12,color=AZUL_ESC,pad=12)
ax1.set_ylabel('Preco medio por m2 (R$)'); ax1.set_ylim(0,max(medias)*1.2)
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)
ax1.grid(True,axis='y',alpha=.25,zorder=0)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,p:f'{int(x):,}'.replace(',','.')))
# Dir: scatter preco x area
for t,cor,lbl in [('casa',TEAL,'Casa'),('apartamento',AZUL,'Apartamento')]:
    s=tip[tip['tipo']==t]
    ax2.scatter(s['area_m2'],s['preco_total'],color=cor,s=70,alpha=.75,
                edgecolor='white',linewidth=1,label=lbl,zorder=3)
ax2.set_title('Preco x area',fontsize=12,color=AZUL_ESC,pad=12)
ax2.set_xlabel('Area (m2)'); ax2.set_ylabel('Preco total'); ax2.legend(fontsize=10)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)
ax2.grid(True,alpha=.25,zorder=0)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,p:f'{x/1000:,.0f}k'.replace(',','.')))
plt.tight_layout(); plt.show()

**Apartamento custa cerca de 46% mais por m² que casa** (medianas de **R\$ 7.624 vs. R\$ 5.217**). A diferença é descritiva da amostra e pode refletir composição por bairro, tipologia, área e padrão dos imóveis; não isola o efeito causal de morar em apartamento.

O imóvel típico da amostra custa por volta de **R\$ 540 mil**, e o anúncio mais acessível dentro do grupo típico é próximo de **R\$ 205 mil**. Esses valores são preços pedidos, não transações concluídas.

Lucas do Rio Verde tinha uma estimativa oficial de **99.291 habitantes em 2026**. Para contexto regional, usamos o FipeZap (abril de 2026) como referência para Cuiabá e Brasil. A população ajuda a contextualizar a comparação, mas não explica sozinha as diferenças de preço.

In [ ]:
lucas=tip['preco_m2'].median(); cuiaba=6931; brasil=9769
print('=== PRECO/M2: LUCAS vs CAPITAL vs BRASIL ===')
print(f'  Lucas do Rio Verde (mediana): R$ {lucas:>6,.0f}/m2'.replace(',','.'))
print(f'  Cuiaba - capital (FipeZap):   R$ {cuiaba:>6,.0f}/m2'.replace(',','.'))
print(f'  Brasil - 56 cidades (FipeZap):R$ {brasil:>6,.0f}/m2'.replace(',','.'))
print()
print(f'  Lucas vs Cuiaba:  {(lucas/cuiaba-1)*100:+.0f}%')
print(f'  Lucas vs Brasil:  {(lucas/brasil-1)*100:+.0f}%')

In [ ]:
fig,ax=plt.subplots(figsize=(11,5.5))
loc=['Lucas do\nRio Verde','Cuiaba\n(capital)','Brasil\n(56 cidades)']
val=[lucas,cuiaba,brasil]
ax.bar(loc,val,color=[LARANJA,AZUL,CINZA],width=.55,zorder=3)
for x,y in enumerate(val):
    ax.annotate(f'R$ {y:,.0f}'.replace(',','.'),(x,y),textcoords='offset points',
                xytext=(0,8),ha='center',fontsize=13,fontweight='bold',color=AZUL_ESC)
ax.set_title('Lucas do Rio Verde cobra 90% do m2 da capital do estado\nPreco por m2 - residencial',
             fontsize=13,color=AZUL_ESC,pad=15)
ax.set_ylabel('Preco por m2 (R$)'); ax.set_ylim(0,max(val)*1.18)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.grid(True,axis='y',alpha=.25,zorder=0)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,p:f'{int(x):,}'.replace(',','.')))
plt.tight_layout(); plt.show()

Na amostra coletada, a mediana do preço anunciado por m² em Lucas fica aproximadamente **10% abaixo da referência do FipeZap para Cuiabá**. Isso mostra que os preços anunciados locais estão próximos à referência da capital, mas as metodologias são distintas e a comparação não equivale a um índice oficial de preço para Lucas.

A série de quatro projetos oferece contexto para investigar a questão — crescimento populacional, oferta de habitação social, preços e acessibilidade. **A associação entre esses temas não demonstra, por si só, que a pressão populacional tenha causado os preços observados.**

## 7. Limitações

1. **Retrato do momento, não série histórica.** Dados coletados em outubro de 2026; anúncios antigos não ficam necessariamente arquivados.
2. **Preço de anúncio, não de fechamento.** O valor pedido pode divergir do valor final de venda.
3. **Possível sobreposição entre fontes.** Três linhas integralmente duplicadas foram removidas. Combinações iguais de bairro, tipo, área e preço em fontes diferentes são candidatas à revisão manual; não foram removidas automaticamente.
4. **Área informada na fonte.** Não houve medição independente da área construída.
5. **Médias por bairro com amostras variadas.** O mapa inclui bairros com 5+ anúncios; algumas amostras são pequenas e indicativas.
6. **Comparação regional com fontes distintas.** Lucas usa anúncios locais; Cuiabá e Brasil, FipeZap (metodologia própria). A comparação indica ordem de grandeza, não uma medição idêntica.

## 8. Síntese executiva

Na amostra deduplicada de **233 anúncios** (dos quais **186 são classificados como típicos**), o preço mediano é de cerca de **R\$ 6.212/m²**. O imóvel típico anunciado fica perto de **R\$ 540 mil**, e o menor preço total observado no grupo típico é de **R\$ 205 mil**.

Os preços por m² variam por bairro. Parque das Emas e Bandeirantes ficam entre as médias mais altas, enquanto Jardim Imperial e Jardim Primavera estão entre as mais baixas. A diferença entre as médias dos bairros extremos elegíveis é de aproximadamente **119% em relação ao mais barato**. Apartamentos têm mediana de preço por m² cerca de **46% acima** da mediana das casas.

A mediana local de preço anunciado por m² fica cerca de 10% abaixo da referência do FipeZap para Cuiabá, mas a comparação combina fontes diferentes e não representa um índice oficial de preço de Lucas do Rio Verde.

A análise é descritiva: retrata anúncios coletados em outubro de 2026, não vendas fechadas, e pode manter sobreposição entre anúncios de fontes diferentes. A conexão com os outros projetos ajuda a formular hipóteses sobre habitação, mas não comprova causalidade entre crescimento, oferta e preços.

---

*Dataset de 233 anúncios (186 típicos) de 13 fontes, coletados em outubro de 2026. A base, o código e as outras camadas da análise estão no GitHub.*